# 🩺 OJAAI — Donut Handwriting Parser: Self-Contained Training on Google Colab

**No file upload needed.** This notebook:
1. Installs all required packages
2. Downloads the Kaggle prescription dataset directly
3. Inlines all training code from the OJAAI project
4. Fine-tunes the Donut model on a T4 GPU
5. Packages the weights for you to download

---
### ⚡ Before running:
- Set runtime to **GPU** → Runtime → Change runtime type → **T4 GPU**
- Add your **Kaggle API credentials** in the cell below


## Step 0: Configure Kaggle API Credentials

In [ ]:
import os

# ── FILL IN YOUR KAGGLE CREDENTIALS BELOW ─────────────────────────────────────
KAGGLE_USERNAME = "sayan17646"  # <-- change this
KAGGLE_KEY      = "KGAT_cfebe22bda524e79fddeea7ca0281758"   # <-- change this (from kaggle.com/account)
# ──────────────────────────────────────────────────────────────────────────────

os.environ["KAGGLE_USERNAME"] = KAGGLE_USERNAME
os.environ["KAGGLE_KEY"]      = KAGGLE_KEY

print(f"✅ Kaggle credentials set for user: {KAGGLE_USERNAME}")

## Step 1: Install Dependencies

In [ ]:
import os
import shutil
from pathlib import Path

# 1. Clone the Hugging Face dataset containing the annotated prescriptions
print("Cloning Hugging Face dataset chinmays18/medical-prescription-dataset...")
repo_url = "https://huggingface.co/datasets/chinmays18/medical-prescription-dataset"
repo_dir = Path("/content/medical-prescription-dataset")

if repo_dir.exists():
    print("Dataset repository already cloned. Skipping clone.")
else:
    !git clone {repo_url} {repo_dir}
    print("✅ Dataset cloned successfully.")

## Step 2: Verify GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"✅ GPU detected: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ No GPU found! Go to Runtime → Change runtime type → T4 GPU and restart.")
    raise RuntimeError("GPU required for training. Please enable GPU runtime.")

## Step 3: Download Dataset from Kaggle

In [ ]:
import os
import shutil
from pathlib import Path

# 1. Clone the Hugging Face dataset containing the annotated prescriptions
print("Cloning Hugging Face dataset chinmays18/medical-prescription-dataset...")
repo_url = "https://huggingface.co/datasets/chinmays18/medical-prescription-dataset"
repo_dir = Path("/content/medical-prescription-dataset")

if repo_dir.exists():
    print("Dataset repository already cloned. Skipping clone.")
else:
    !git clone {repo_url} {repo_dir}
    print("✅ Dataset cloned successfully.")

## Step 4: Prepare Training Data (Inline Dataset Builder)

In [ ]:
import os
import json
import re
import shutil
from pathlib import Path
from typing import Any

# Setup directories
TRAIN_DIR = Path("/content/donut_train_data")
IMAGES_DIR = TRAIN_DIR / "images"
TRAIN_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

FIELD_NAMES = (
    "doctor_name", "clinic_name", "clinic_address", "patient_name", "patient_age",
    "patient_gender", "date", "diagnosis", "medications", "signature",
)
FIELD_BOUNDARY_RE = "|".join(f"{name}:" for name in FIELD_NAMES)
DOSAGE_RE = re.compile(
    r"(?P<drug>[A-Za-z][A-Za-z0-9 ./'-]*?)\s+"
    r"(?P<value>\d+(?:\.\d+)?)\s*"
    r"(?P<unit>mg|mcg|ml|g|iu|units?|puffs?|drops?)\b",
    re.I,
)
NON_DRUG_INSTRUCTION_STARTS = {"after", "as", "at", "before", "every", "for", "if", "in", "on", "once", "take", "three", "twice", "with"}
FREQUENCY_MAP = {
    "once daily": 1, "od": 1, "every 24 hours": 1,
    "every 12 hours": 2, "twice daily": 2, "bd": 2, "bid": 2,
    "three times daily": 3, "tds": 3, "tid": 3,
    "four times daily": 4, "qds": 4, "qid": 4,
    "as needed": 0, "sos": 0, "prn": 0,
}

def clean_text(value: str) -> str:
    value = re.sub(r"</?s(?:_ocr)?>", " ", value)
    value = re.sub(r"\s+", " ", value)
    return value.strip(" \t\r\n:-")

def parse_header_field(gt_str: str, field_name: str) -> str:
    pattern = rf"\b{re.escape(field_name)}:\s*(.*?)\s*(?={FIELD_BOUNDARY_RE}|</s>|$)"
    match = re.search(pattern, gt_str, re.I | re.S)
    return clean_text(match.group(1)) if match else ""

def frequency_to_per_day(frequency: str | None) -> int | None:
    if not frequency:
        return None
    normalized = clean_text(frequency).lower()
    for key, value in FREQUENCY_MAP.items():
        if key in normalized:
            return value
    return None

def parse_gt_medications(gt_str: str) -> list[dict[str, Any]]:
    meds_match = re.search(
        rf"\bmedications:\s*(.*?)\s*(?=signature:|{FIELD_BOUNDARY_RE}|</s>|$)",
        gt_str,
        re.I | re.S,
    )
    if not meds_match:
        return []

    meds_block = clean_text(meds_match.group(1))
    raw_items = [item.strip() for item in re.split(r"\s+-\s+", f" {meds_block}") if item.strip()]
    medications = []
    pending = None

    for item in raw_items:
        item = clean_text(item)
        dosage_match = DOSAGE_RE.search(item)
        if dosage_match:
            if pending:
                medications.append(pending)
            pending = {
                "drug_name": clean_text(dosage_match.group("drug")),
                "dosage_value": float(dosage_match.group("value")),
                "dosage_unit": dosage_match.group("unit").lower(),
                "frequency": None,
                "freq_per_day": None,
                "duration_days": None,
                "route": "oral",
            }
            trailing = clean_text(item[dosage_match.end():])
            if trailing:
                pending["frequency"] = trailing
                pending["freq_per_day"] = frequency_to_per_day(trailing)
            continue

        first_word = item.split()[0].lower() if item.split() else ""
        if pending and first_word in NON_DRUG_INSTRUCTION_STARTS:
            pending["frequency"] = item
            pending["freq_per_day"] = frequency_to_per_day(item)
            continue

        if pending:
            medications.append(pending)
        pending = {
            "drug_name": item,
            "dosage_value": None,
            "dosage_unit": None,
            "frequency": None,
            "freq_per_day": None,
            "duration_days": None,
            "route": "oral",
        }

    if pending:
        medications.append(pending)
    return [med for med in medications if med.get("drug_name")]

def compact_ground_truth(value: Any) -> Any:
    if isinstance(value, dict):
        compacted = {}
        for key, item in value.items():
            cleaned = compact_ground_truth(item)
            if cleaned not in (None, "", [], {}):
                compacted[key] = cleaned
        return compacted
    if isinstance(value, list):
        return [item for item in (compact_ground_truth(item) for item in value) if item not in (None, "", [], {})]
    if isinstance(value, str):
        return clean_text(value)
    return value

def has_non_empty_labels(ground_truth: dict[str, Any]) -> bool:
    if any(ground_truth.get(key) for key in ("doctor_name", "clinic_name", "patient_name", "patient_age", "prescription_date")):
        return True
    meds = ground_truth.get("medications")
    return isinstance(meds, list) and any(isinstance(med, dict) and med.get("drug_name") for med in meds)

def validate_ground_truth(ground_truth: dict[str, Any], source: str) -> None:
    if not has_non_empty_labels(ground_truth):
        raise ValueError(f"{source}: empty Donut labels; refusing to train on null-only ground_truth")
    for idx, med in enumerate(ground_truth.get("medications", [])):
        if not isinstance(med, dict) or not med.get("drug_name"):
            raise ValueError(f"{source}: medication #{idx + 1} is missing drug_name")


In [ ]:
# Format Hugging Face annotations into metadata.jsonl. Refuse image-only / empty-label datasets.
repo_dir = Path("/content/medical-prescription-dataset")
train_ann_dir = repo_dir / "train" / "annotations"
train_img_dir = repo_dir / "train" / "images"

if not train_ann_dir.exists():
    train_ann_dir = repo_dir / "test" / "annotations"
    train_img_dir = repo_dir / "test" / "images"

if not train_ann_dir.exists() or not train_img_dir.exists():
    raise FileNotFoundError("Annotated Hugging Face dataset folders were not found. Do not train on unannotated Kaggle images.")

json_files = sorted(train_ann_dir.glob("*.json"))
print(f"Processing {len(json_files)} annotations from {train_ann_dir}...")

entries = []
for file_path in json_files:
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    gt_str = data.get("ground_truth", "")
    if not gt_str:
        raise ValueError(f"{file_path.name}: missing source ground_truth")

    doctor_name = parse_header_field(gt_str, "doctor_name")
    reg_match = re.search(r"\b(?:mci|reg|nmc)[\s.:-]*([A-Z0-9/-]+)", doctor_name, re.I)
    structured_gt = compact_ground_truth({
        "doctor_name": doctor_name,
        "doctor_reg": reg_match.group(1) if reg_match else None,
        "clinic_name": parse_header_field(gt_str, "clinic_name"),
        "patient_name": parse_header_field(gt_str, "patient_name"),
        "patient_age": parse_header_field(gt_str, "patient_age"),
        "patient_gender": parse_header_field(gt_str, "patient_gender"),
        "prescription_date": parse_header_field(gt_str, "date"),
        "diagnosis": parse_header_field(gt_str, "diagnosis"),
        "medications": parse_gt_medications(gt_str),
    })
    validate_ground_truth(structured_gt, file_path.name)

    img_name = file_path.stem + ".png"
    src_img = train_img_dir / img_name
    if not src_img.exists():
        raise FileNotFoundError(f"{file_path.name}: matching image not found: {src_img}")

    dest_img = IMAGES_DIR / img_name
    shutil.copy2(src_img, dest_img)
    entries.append({
        "file_name": f"images/{img_name}",
        "ground_truth": json.dumps(structured_gt, ensure_ascii=False, sort_keys=True),
    })

if not entries:
    raise RuntimeError("No validated Donut metadata entries were generated.")

metadata_path = TRAIN_DIR / "metadata.jsonl"
with open(metadata_path, "w", encoding="utf-8") as f:
    for entry in entries:
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

print(f"? Preprocessing complete. Wrote {len(entries)} validated entries to {metadata_path}")
print("\nSample training records:")
for entry in entries[:3]:
    print(json.dumps({"file_name": entry["file_name"], "ground_truth": json.loads(entry["ground_truth"])}, indent=2, ensure_ascii=False))


## Step 5: Define Donut Dataset Class & Training Code

In [ ]:
import os
import json
import logging
import re
from pathlib import Path
from typing import Any

import torch
from torch.utils.data import Dataset
from PIL import Image
from transformers import (
    DonutProcessor,
    VisionEncoderDecoderModel,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
)

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
logger = logging.getLogger(__name__)

TASK_START_TOKEN = "<s_rx>"
EOS_TOKEN = "</s>"

def json_to_donut_sequence(obj: Any) -> str:
    obj = compact_ground_truth(obj)
    if isinstance(obj, dict):
        result = ""
        for k, v in obj.items():
            v = compact_ground_truth(v)
            if v in (None, "", [], {}):
                continue
            result += f"<s_{k}>" + json_to_donut_sequence(v) + f"</s_{k}>"
        return result
    if isinstance(obj, list):
        result = ""
        for item in obj:
            item = compact_ground_truth(item)
            if item not in (None, "", [], {}):
                result += "<s_el>" + json_to_donut_sequence(item) + "</s_el>"
        return result
    return str(obj) if obj is not None else ""

def build_target_sequence(gt: dict[str, Any]) -> str:
    gt = compact_ground_truth(gt)
    validate_ground_truth(gt, "target_sequence")
    body = json_to_donut_sequence(gt)
    if not body.strip():
        raise ValueError("Cannot train on empty Donut target sequence")
    return f"{TASK_START_TOKEN}{body}{EOS_TOKEN}"

class PrescriptionDataset(Dataset):
    """PyTorch Dataset for prescription images to structured JSON via Donut."""
    def __init__(self, dataset_path, processor, max_length=512, ignore_id=-100):
        super().__init__()
        self.dataset_path = Path(dataset_path)
        self.processor = processor
        self.max_length = max_length
        self.ignore_id = ignore_id
        metadata_file = self.dataset_path / "metadata.jsonl"
        if not metadata_file.exists():
            raise FileNotFoundError(f"metadata.jsonl not found at {metadata_file}")
        self.samples = []
        with open(metadata_file, "r", encoding="utf-8") as f:
            for line_no, line in enumerate(f, start=1):
                if not line.strip():
                    continue
                entry = json.loads(line)
                gt = json.loads(entry["ground_truth"])
                gt = compact_ground_truth(gt)
                validate_ground_truth(gt, f"metadata.jsonl:{line_no}")
                entry["ground_truth"] = json.dumps(gt, ensure_ascii=False, sort_keys=True)
                self.samples.append(entry)
        if not self.samples:
            raise ValueError("No valid training samples found")
        logger.info(f"Loaded {len(self.samples)} validated training samples from {dataset_path}")

    def __len__(self):
        return len(self.samples)

    def target_sequence(self, idx):
        return build_target_sequence(json.loads(self.samples[idx]["ground_truth"]))

    def __getitem__(self, idx):
        sample = self.samples[idx]
        img_path = self.dataset_path / sample["file_name"]
        if not img_path.exists():
            raise FileNotFoundError(f"Image not found: {img_path}")
        image = Image.open(img_path).convert("RGB")
        pixel_values = self.processor(image, return_tensors="pt").pixel_values.squeeze(0)
        target_seq = self.target_sequence(idx)
        labels = self.processor.tokenizer(
            target_seq,
            max_length=self.max_length,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        ).input_ids.squeeze(0)
        labels_for_loss = labels.clone()
        labels_for_loss[labels == self.processor.tokenizer.pad_token_id] = self.ignore_id
        return {"pixel_values": pixel_values, "labels": labels_for_loss}

def decoded_labels_to_text(labels, tokenizer, ignore_id=-100):
    labels = labels.detach().clone()
    labels[labels == ignore_id] = tokenizer.pad_token_id
    return tokenizer.decode(labels, skip_special_tokens=False).replace(tokenizer.pad_token or "", "").strip()

print("? Dataset class and helper functions defined.")


## Step 6: Load Base Donut Model & Configure Special Tokens

In [ ]:
BASE_MODEL = "naver-clova-ix/donut-base"

print(f"Loading base model: {BASE_MODEL}")
print("This may take a few minutes to download (~700MB)...")

processor = DonutProcessor.from_pretrained(BASE_MODEL)
model = VisionEncoderDecoderModel.from_pretrained(BASE_MODEL)

# Add prescription-specific structural tokens
special_tokens = [
    "<s>", "</s>",
    "<s_rx>", "</s_rx>",
    "<s_doctor_name>", "</s_doctor_name>",
    "<s_doctor_reg>", "</s_doctor_reg>",
    "<s_clinic_name>", "</s_clinic_name>",
    "<s_patient_name>", "</s_patient_name>",
    "<s_patient_age>", "</s_patient_age>",
    "<s_patient_gender>", "</s_patient_gender>",
    "<s_prescription_date>", "</s_prescription_date>",
    "<s_diagnosis>", "</s_diagnosis>",
    "<s_medications>", "</s_medications>",
    "<s_el>", "</s_el>",
    "<s_drug_name>", "</s_drug_name>",
    "<s_dosage_value>", "</s_dosage_value>",
    "<s_dosage_unit>", "</s_dosage_unit>",
    "<s_frequency>", "</s_frequency>",
    "<s_freq_per_day>", "</s_freq_per_day>",
    "<s_duration_days>", "</s_duration_days>",
    "<s_route>", "</s_route>",
    "<s_instructions>", "</s_instructions>"
]

num_added = processor.tokenizer.add_special_tokens({"additional_special_tokens": special_tokens})
print(f"Added {num_added} special tokens to tokenizer.")

# Resize decoder embedding to cover new tokens
model.decoder.resize_token_embeddings(len(processor.tokenizer))

# Configure generation params
model.config.pad_token_id = processor.tokenizer.pad_token_id
model.config.decoder_start_token_id = processor.tokenizer.convert_tokens_to_ids(["<s>"])[0]

print(f"✅ Model loaded. Tokenizer vocab size: {len(processor.tokenizer)}")
print(f"   Model parameter count: {sum(p.numel() for p in model.parameters()):,}")

## Step 7: Create Dataset & Configure Trainer

In [ ]:
TRAIN_DIR = Path("/content/donut_train_data")

USE_DRIVE = False
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = Path("/content/drive/MyDrive/donut-rx-finetuned")
else:
    OUTPUT_DIR = Path("/content/donut-rx-finetuned")

EPOCHS        = 30
BATCH_SIZE    = 1
GRAD_ACCUM    = 8
LEARNING_RATE = 2e-5
MAX_LENGTH    = 512

train_dataset = PrescriptionDataset(
    dataset_path=TRAIN_DIR,
    processor=processor,
    max_length=MAX_LENGTH,
)
print(f"? Training dataset: {len(train_dataset)} samples")
print("\nValidated sample training records:")
for entry in train_dataset.samples[:3]:
    print(json.dumps({"file_name": entry["file_name"], "ground_truth": json.loads(entry["ground_truth"])}, indent=2, ensure_ascii=False))

first_target = train_dataset.target_sequence(0)
if "<s_medications></s_medications>" in first_target or re.search(r"<s_[^>]+>\s*</s_[^>]+>", first_target):
    raise ValueError(f"Empty tags found in target sequence: {first_target}")
first_item = train_dataset[0]
decoded_target = decoded_labels_to_text(first_item["labels"], processor.tokenizer, train_dataset.ignore_id)
print("\nDecoded target sequence before training:")
print(decoded_target)
if "<s_iitcdip>" in decoded_target:
    raise ValueError("Decoded target contains Donut base task token <s_iitcdip>; expected <s_rx>")

training_args = Seq2SeqTrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    weight_decay=0.01,
    warmup_steps=50,
    logging_steps=5,
    save_steps=100,
    save_total_limit=3,
    predict_with_generate=True,
    fp16=True,
    dataloader_num_workers=2,
    report_to="none",
    load_best_model_at_end=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)
print(f"? Trainer configured. Starting training for {EPOCHS} epochs only after label validation passed...")


## Step 8: 🚀 Train!

In [ ]:
import time

start_time = time.time()
print("🚀 Training started...")
print(f"   GPU: {torch.cuda.get_device_name(0)}")
print(f"   Dataset size: {len(train_dataset)} images")
print(f"   Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"   Epochs: {EPOCHS}")
print("")

train_result = trainer.train()

elapsed = time.time() - start_time
print(f"")
print(f"✅ Training complete! Total time: {elapsed/60:.1f} minutes")
print(f"   Final loss: {train_result.training_loss:.4f}")

# Save model and processor
print(f"\nSaving model to {OUTPUT_DIR}...")
trainer.save_model(str(OUTPUT_DIR))
processor.save_pretrained(str(OUTPUT_DIR))
print("✅ Model saved!")

## Step 9: Run a Quick Inference Test

In [ ]:
import re

def has_structured_output(parsed: dict) -> bool:
    if not isinstance(parsed, dict) or not parsed:
        return False
    if any(parsed.get(key) for key in ("doctor_name", "clinic_name", "patient_name", "patient_age", "prescription_date")):
        return True
    meds = parsed.get("medications")
    return isinstance(meds, list) and any(isinstance(med, dict) and med.get("drug_name") for med in meds)

sample_images = list((TRAIN_DIR / "images").glob("*.png"))
if not sample_images:
    raise FileNotFoundError("No images found for inference test.")

test_image_path = sample_images[0]
print(f"Testing inference on: {test_image_path.name}")
model.eval()
model.to("cuda")
test_image = Image.open(test_image_path).convert("RGB")
pixel_values = processor(test_image, return_tensors="pt").pixel_values.to("cuda")
decoder_input_ids = processor.tokenizer(TASK_START_TOKEN, add_special_tokens=False, return_tensors="pt").input_ids.to("cuda")
eos_token_id = processor.tokenizer.convert_tokens_to_ids(EOS_TOKEN)

with torch.no_grad():
    outputs = model.generate(
        pixel_values,
        decoder_input_ids=decoder_input_ids,
        max_length=512,
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=eos_token_id,
        use_cache=True,
        num_beams=1,
        do_sample=False,
    )

output_seq = processor.batch_decode(outputs, skip_special_tokens=False)[0]
output_seq = output_seq.replace(processor.tokenizer.pad_token or "", "").strip()
if output_seq.startswith(TASK_START_TOKEN):
    output_seq = output_seq[len(TASK_START_TOKEN):]
output_seq = output_seq.replace(processor.tokenizer.eos_token or "", "").replace(EOS_TOKEN, "").strip()
print(f"\nRaw output sequence:\n{output_seq}")

if not output_seq or output_seq == "<s_iitcdip>":
    raise RuntimeError(f"Inference failed: model returned base/empty task output {output_seq!r}")

parsed = processor.token2json(output_seq)
print("\nParsed JSON output:")
print(json.dumps(parsed, indent=2, ensure_ascii=False))
if not has_structured_output(parsed):
    raise RuntimeError("Inference returned empty structured output; weights must not be packaged.")
print("? Inference returned non-empty structured output.")


## Step 10: Package & Download Model Weights

In [ ]:
# Package weights only if the inference validation cell produced non-empty structured output.
import shutil

if "parsed" not in globals() or not has_structured_output(parsed):
    raise RuntimeError("Refusing to package weights because inference did not return non-empty structured output.")

print("Packaging verified model weights...")
shutil.make_archive("/content/donut_rx_weights", "zip", "/content/donut-rx-finetuned")
print("? Model weights packaged: /content/donut_rx_weights.zip")
size_mb = os.path.getsize("/content/donut_rx_weights.zip") / 1e6
print(f"   Size: {size_mb:.1f} MB")
print("\n?? Download donut_rx_weights.zip and place the unzipped folder into: c:/Users/USER/Desktop/OJAAI/models/donut-rx/")


---
## ✅ Done!

After downloading `donut_rx_weights.zip`:

1. Extract it
2. Place the contents into `c:/Users/USER/Desktop/OJAAI/models/donut-rx/`
3. The OJAAI pipeline will automatically use the fine-tuned weights as Tier 3 fallback for handwritten prescriptions

**Model location the pipeline expects:** `./models/donut-rx/`